# 05 · Sensors: from the true state to what PX4 sees

PX4 never sees the simulator's true state, only sensor readings made from it. This notebook builds those readings: the accelerometer's specific force, the gyro, noise, and the H-FLOW optical-flow and range sensor, including the two bugs fixed on 1 Oct.

In [1]:
# Setup: make the repository importable and load the V3 airframe used throughout these notebooks
import sys, math, pathlib
ROOT = pathlib.Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
from airframe_designer.geometry.airframe import Airframe

np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({"figure.figsize": (7, 3.6), "axes.grid": True, "grid.alpha": 0.3})

AF_PATH = ROOT / "airframes" / "atlas_v3_v34_foils_50_65_50_tuned.json"
af = Airframe.load(AF_PATH)
af.resolve_mass()
G = 9.80665
print(f"{af.name}: {af.mass.mass:.2f} kg, {len(af.active_rotors())} fans, hover pitch {af.hover_pitch_deg} deg, parked {af.landed_pitch_deg} deg")

ATLAS_V3_V34_FOILS_50_65_50_TUNED: 11.83 kg, 9 fans, hover pitch 23.85 deg, parked 23.85 deg


In [2]:
from airframe_designer.dynamics.rigid_body import RigidBody
from airframe_designer.dynamics.quaternion import q_to_rotmat, q_from_euler
from airframe_designer.sensors.models import SensorSuite
rb = RigidBody(af); rb.set_motor_commands(np.zeros(rb.rotors.n))
for _ in range(3000):
    rb.step(1e-3, detail=False)

## 1. The accelerometer measures specific force

**Theory.**

$$\mathbf f^B = R(\mathbf q)^{\mathsf T}\Big(\frac{\mathbf v_{k+1}-\mathbf v_k}{\Delta t} - g\,\hat{\mathbf e}_3\Big).$$

The simulator derives it from the motion it actually integrated, so the reading is exactly consistent with the trajectory.

> **Note: what an accelerometer feels**
>
> An accelerometer cannot feel gravity. It feels everything that stops it from falling. Sitting on the floor it reads "pushed up by 1 g", $(0, 0, -g)$ in level axes; in free fall it reads zero. Because the reading is in the sensor's own axes, a tilted aircraft at rest shows part of that 1 g on its x axis.

In [3]:
R = q_to_rotmat(rb.q)
print("simulator accel_body at rest      :", rb.accel_body)
print("R^T (0, 0, -g), the expected value :", R.T @ [0, 0, -G])
q_h = q_from_euler(0, math.radians(af.hover_pitch_deg), 0)
print("in a perfect hover (structural axes):", q_to_rotmat(q_h).T @ [0, 0, -G], " (x feels part of the thrust)")

simulator accel_body at rest      : [ 3.8935  0.0042 -9.111 ]
R^T (0, 0, -g), the expected value : [ 3.8536  0.0041 -9.0178]
in a perfect hover (structural axes): [ 3.9653  0.     -8.9692]  (x feels part of the thrust)


## 2. Gyro and the full HIL_SENSOR message

**Theory.** $\boldsymbol\omega_{\text{gyro}} = \boldsymbol\omega^B + \mathbf b_g + \mathbf n_g + \boldsymbol\omega_{\text{vib}}$: the true rate plus a constant bias, white noise, and the fan vibration model. The magnetometer is $R^{\mathsf T}\mathbf m^N$ plus noise; the barometer is ISA pressure at the current altitude plus noise.

In [4]:
ss = SensorSuite(seed=1); ss.set_airframe(af)
acc, gyr = [], []
for k in range(2000):
    m = ss.hil_sensor(rb, k * 4000)
    acc.append([m["xacc"], m["yacc"], m["zacc"]]); gyr.append([m["xgyro"], m["ygyro"], m["zgyro"]])
acc, gyr = np.array(acc), np.array(gyr)
print("accel mean", acc.mean(0), " std", acc.std(0))
print("gyro  mean", gyr.mean(0), " std", gyr.std(0))
print("one message:", {k: (round(v, 3) if isinstance(v, float) else v) for k, v in m.items()})

accel mean [ 3.8925  0.0046 -9.1112]  std [0.0196 0.0197 0.0198]
gyro  mean [-0.     -0.     -0.0001]  std [0.002 0.002 0.002]
one message: {'time_usec': 7996000, 'xacc': 3.863, 'yacc': -0.009, 'zacc': -9.133, 'xgyro': -0.0, 'ygyro': 0.001, 'zgyro': -0.001, 'xmag': -0.039, 'ymag': 0.004, 'zmag': 0.502, 'abs_pressure': 1012.023, 'diff_pressure': 0.0, 'pressure_alt': 10.221, 'temperature': 14.933, 'fields_updated': 7167, 'id': 0}


> **Note: truth versus estimate**
>
> PX4's estimator (EKF2) builds its own state from these readings: attitude, velocity, position, gyro and accelerometer biases, magnetic fields and wind, about 24 numbers. Almost every HITL problem on 30 Sep was a gap between that estimate and the simulator's truth (a stored accelerometer offset, noisy flow), not a problem with the true motion. Comparing estimate with truth is the most useful diagnostic the simulator offers.

## 3. The H-FLOW: optical flow and range

**Theory.** A downward camera at height $d$ (along its optical axis) over a flat floor sees the image centre move by

$$\text{pixel}_x = \Delta\theta_x - \frac{v_y\,\Delta t}{d},\qquad \text{pixel}_y = \Delta\theta_y + \frac{v_x\,\Delta t}{d},$$

with $\mathbf v$ the lens velocity and $\Delta\boldsymbol\theta$ the rotation over the window, both in the sensor frame. PX4 subtracts the rotation using a gyro and recovers $v = d\cdot(\text{flow} - \text{gyro})/\Delta t$.

The as-built mount: fixed to the frame, tilted 20.4°, right below the Pixhawk, so in the 23.85° hover it looks 3.45° off vertical.

In [5]:
from airframe_designer.sensors.flow import FlowSensor, ekf2_params
cfg = dict(af.design["flow_sensor"]); cfg["enabled"] = True
print("flow sensor config:", {k: cfg[k] for k in ("pos", "mount", "extra_pitch_deg", "rate", "flow_noise", "range_noise")})
print("EKF2 settings that match it:")
for k, v in ekf2_params(af, cfg).items():
    print(f"   {k:18s} {v}")

flow sensor config: {'pos': [-0.015, 0.0, 0.062], 'mount': 'structure', 'extra_pitch_deg': 20.4, 'rate': 50, 'flow_noise': 0.02, 'range_noise': 0.01}
EKF2 settings that match it:
   EKF2_RNG_PITCH     0.0602
   EKF2_GPS_CTRL      0
   EKF2_OF_CTRL       1
   EKF2_RNG_CTRL      1
   EKF2_HGT_REF       2
   EKF2_OF_POS_X      0.066
   EKF2_OF_POS_Y      0.004
   EKF2_OF_POS_Z      -0.048
   EKF2_RNG_POS_X     0.066
   EKF2_RNG_POS_Y     0.004
   EKF2_RNG_POS_Z     -0.048
   EKF2_IMU_POS_X     0.0
   EKF2_IMU_POS_Y     0.0
   EKF2_IMU_POS_Z     0.0
   SENS_FLOW_ROT      0
   SENS_FLOW_MINHGT   0.08
   SENS_FLOW_MAXHGT   8.0
   COM_ARM_WO_GPS     1
   EKF2_OF_DELAY      0.0
   EKF2_RNG_DELAY     0.0


> **Note: the two flow bugs fixed on 1 Oct**
>
> 1. **Noise 7 times too large.** `flow_noise` is the noise of one reading (0.02 rad/s). The code used to scale it by $\sqrt{\Delta t}$ as if it were a noise density, which at 50 readings a second made each one 0.14 rad/s: about 0.37 m/s of fake motion at 2.6 m. EKF2 kept nudging its heading to explain that noise. Now it is `noise * dt`.
> 2. **Gyro in the wrong frame.** The message now carries the sensor's own gyro integral, so PX4 does not fall back to the raw, unrotated board gyro (notebook 01, section 5).
>
> The cell measures both: the decoded noise per reading, and a pure turn on the tilted mount. The lens sits about 8 cm from the CG, so a turn really moves it ($\boldsymbol\omega\times\mathbf r$); the decoded velocity must equal exactly that lens motion, with no rotation leaking in.

In [6]:
from types import SimpleNamespace
fs = FlowSensor(af, cfg, np.random.default_rng(3))
def fake_sim(vel_ned, rates, pitch_deg, height):
    Rm = q_to_rotmat(q_from_euler(0, math.radians(pitch_deg), 0))
    return SimpleNamespace(rotmat=Rm, rates=np.asarray(rates, float), vel=np.asarray(vel_ned, float), pos=np.array([0, 0, -height]))
dt = 1 / fs.rate
rates = []
for _ in range(3000):
    s = fake_sim([0, 0, 0], [0, 0, 0], af.hover_pitch_deg, 1.5)
    fs.step(s, dt); of, ds = fs.messages(s, 0, dt)
    rates.append(of["integrated_x"] / dt)
print(f"flow noise per reading: {np.std(rates):.4f} rad/s (configured {cfg['flow_noise']})")

fs2 = FlowSensor(af, dict(cfg, flow_noise=0, range_noise=0), np.random.default_rng(1))
s = fake_sim([0, 0, 0], [0.2, -0.1, 0.5], af.hover_pitch_deg, 1.5)
fs2.step(s, dt); of, ds = fs2.messages(s, 0, dt)
flow = -np.array([of["integrated_x"], of["integrated_y"]]); gyro = -np.array([of["integrated_xgyro"], of["integrated_ygyro"]])
comp = flow - gyro; d = of["distance"]
decoded = np.array([-d * comp[1] / dt, d * comp[0] / dt])
lens = (fs2.C.T @ np.cross([0.2, -0.1, 0.5], fs2.r))[:2]          # the lens's own motion from the lever arm, sensor axes
print("pure turn, decoded lens velocity :", np.round(decoded, 5), "m/s   range", round(d, 3), "m")
print("lens motion from omega x r       :", np.round(lens, 5), "m/s   match:", np.allclose(decoded, lens, atol=1e-6))

flow noise per reading: 0.0203 rad/s (configured 0.02)
pure turn, decoded lens velocity : [0.0028 0.0432] m/s   range 1.55 m
lens motion from omega x r       : [0.0028 0.0432] m/s   match: True
